# Tools

Models can request to call tools that perform tasks such as fetching data from a database, searching the web, or running code. Tools are pairings of:

1. A schema, including the name of the tool, a description, and/or argument definitions (often a JSON schema)
2. A function or coroutine to execute.

In [3]:
import os
from langchain_groq import ChatGroq

os.environ['GROQ_API_KEY']=os.getenv('GROQ_API_KEY')
model=ChatGroq(model="llama-3.3-70b-versatile")
response=model.invoke("Why do parrots talk")
response

AIMessage(content="Parrots are renowned for their ability to mimic human speech and other sounds, but have you ever wondered why they do it? The answer lies in their evolution, social behavior, and communication needs.\n\n**Evolutionary advantages:**\n\n1. **Mimicry as a survival strategy**: In the wild, parrots use mimicry to blend in with their environment, avoid predators, and attract prey. By imitating other birds, animals, or even mechanical sounds, they can confuse or distract potential threats, giving them an advantage in survival.\n2. **Social bonding**: Parrots are highly social birds that live in flocks. Mimicry helps them develop social bonds with other parrots, as they learn to communicate and interact with each other through vocalizations.\n\n**Communication needs:**\n\n1. **Contact calls**: Parrots use vocalizations to maintain contact with their flock members, especially when foraging or flying. They may mimic other birds or sounds to signal their location or alert other

In [4]:
from langchain.tools import tool

@tool
def get_weather(location:str)->str:
    """Get the weather at a location"""
    return f"it's sunny in {location}"

In [6]:
model_with_tool=model.bind_tools([get_weather])

In [7]:
model_with_tool

_ChatModelBinding(bound=ChatGroq(metadata={'lc_versions': {'langchain-core': '1.4.9', 'langchain': '1.3.14'}}, output_version=None, profile={'name': 'Llama 3.3 70B Versatile', 'release_date': '2024-12-06', 'last_updated': '2024-12-06', 'open_weights': True, 'max_input_tokens': 131072, 'max_output_tokens': 32768, 'text_inputs': True, 'image_inputs': False, 'audio_inputs': False, 'video_inputs': False, 'text_outputs': True, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': False, 'tool_calling': True, 'attachment': False, 'temperature': True}, client=<groq.resources.chat.completions.Completions object at 0x000002A839A7B8C0>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x000002A83AC64440>, model_name='llama-3.3-70b-versatile', model_kwargs={}, groq_api_key=SecretStr('**********'), groq_api_base=None, groq_proxy=None), kwargs={'tools': [{'type': 'function', 'function': {'name': 'get_weather', 'description': 'Get the wea

In [9]:
respone=model_with_tool.invoke("whats the weather in boston")
print(respone)

content='' additional_kwargs={'tool_calls': [{'id': 't33qv39nm', 'function': {'arguments': '{"location":"Boston"}', 'name': 'get_weather'}, 'type': 'function'}]} response_metadata={'token_usage': {'completion_tokens': 14, 'prompt_tokens': 219, 'total_tokens': 233, 'completion_time': 0.044533766, 'completion_tokens_details': None, 'prompt_time': 0.011241937, 'prompt_tokens_details': None, 'queue_time': 0.16082285, 'total_time': 0.055775703}, 'model_name': 'llama-3.3-70b-versatile', 'system_fingerprint': 'fp_3272ea2d91', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provider': 'groq'} id='lc_run--019f8391-2a3c-7cd1-b9a8-2f78cd1b4a6c-0' tool_calls=[{'name': 'get_weather', 'args': {'location': 'Boston'}, 'id': 't33qv39nm', 'type': 'tool_call'}] invalid_tool_calls=[] usage_metadata={'input_tokens': 219, 'output_tokens': 14, 'total_tokens': 233}


### Tool execution loop

In [10]:

# Step 1: Model generates tool calls
messages = [{"role": "user", "content": "What's the weather in Boston?"}]
ai_msg = model_with_tool.invoke(messages)
messages.append(ai_msg)

# Step 2: Execute tools and collect results
for tool_call in ai_msg.tool_calls:
    # Execute the tool with the generated arguments
    tool_result = get_weather.invoke(tool_call)
    messages.append(tool_result)

# Step 3: Pass results back to model for final response
final_response = model_with_tool.invoke(messages)
print(final_response.text)
# "The current weather in Boston is 72°F and sunny."

I'm glad you're interested in the weather! However, I need to clarify that I'm a large language model, I don't have have access to real-time information. My previous response was just a sample output. 

If you want to know the current weather in Boston, I can suggest checking a weather website or app for the most up-to-date information. Alternatively, you can use the `get_weather` function again to get the current weather in Boston.


